In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier, Pool
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
from sklearn.model_selection import StratifiedKFold, train_test_split

import catboost
print(f'CatBoost version: {catboost.__version__}')
print('Imports OK')

In [ ]:
SEED     = 42
N_TRIALS = 20
np.random.seed(SEED)

BASE_DIR = Path('../../')
DATA_DIR = BASE_DIR / 'data/processed'

matplotlib.rcParams.update({'figure.figsize': (9, 6), 'font.size': 12,
                            'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
df      = pd.read_parquet(DATA_DIR / 'features_train.parquet')
df_test = pd.read_parquet(DATA_DIR / 'features_test.parquet')

feature_cols = [c for c in df.columns if c not in ('SK_ID_CURR', 'TARGET')]

X           = df[feature_cols].values
y           = df['TARGET'].values
X_test      = df_test[feature_cols].values
sk_ids_test = df_test['SK_ID_CURR'].values

print(f'Train: {X.shape}  | Default rate: {y.mean():.2%}')
print(f'Test : {X_test.shape}')

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5, label='Model'):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    auc  = roc_auc_score(y_true, y_prob)
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return dict(Model=label, AUC=round(auc,4),
                N=len(y_true), P=int(y_pred.sum()),
                TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                Recall=round(rec,4), Precision=round(prec,4), F1=round(f1,4))

def plot_roc_curve(y_true, y_prob, label, color, ax, title='ROC Curve'):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{label} (AUC = {auc:.4f})')
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(title); ax.legend()
    return ax

print('Funciones definidas OK')

---
## Optuna — Búsqueda de Hiperparámetros

Usa split 80/20 estratificado como validación con `early_stopping_rounds=50`.
El objetivo penaliza overfitting: `val_AUC - 0.5 * max(0, train_AUC - val_AUC)`.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
print(f'Train optuna: {X_tr.shape} | Val: {X_val.shape}')

def objective(trial):
    params = dict(
        iterations          = 1000,
        learning_rate       = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        depth               = trial.suggest_int('depth', 4, 10),
        l2_leaf_reg         = trial.suggest_float('l2_leaf_reg', 1.0, 30.0, log=True),
        bagging_temperature = trial.suggest_float('bagging_temperature', 0.0, 1.0),
        subsample           = trial.suggest_float('subsample', 0.5, 1.0),
        random_strength     = trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
        auto_class_weights  = 'Balanced',
        eval_metric         = 'AUC',
        early_stopping_rounds = 50,
        random_seed         = SEED,
        verbose             = False,
    )
    model = CatBoostClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    val_auc   = roc_auc_score(y_val,  model.predict_proba(X_val)[:, 1])
    train_auc = roc_auc_score(y_tr,   model.predict_proba(X_tr)[:, 1])
    n_trees   = model.get_best_iteration() + 1

    trial.set_user_attr('n_trees',   n_trees)
    trial.set_user_attr('train_auc', train_auc)
    trial.set_user_attr('val_auc',   val_auc)

    gap = max(0, train_auc - val_auc)
    return val_auc - 0.5 * gap

print(f'Lanzando Optuna — {N_TRIALS} trials...')
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_trial
print(f'\nMejores hiperparámetros:')
for k, v in best.params.items():
    print(f'  {k:<24}: {v}')
print(f'  n_trees (óptimo)    : {best.user_attrs["n_trees"]}')
print(f'\nObjetivo (Val AUC penalizado): {best.value:.4f}')
print(f'Val AUC   : {best.user_attrs["val_auc"]:.4f}')
print(f'Train AUC : {best.user_attrs["train_auc"]:.4f}')

In [ ]:
# Refit en todo el train
best_params = dict(
    iterations          = study.best_trial.user_attrs['n_trees'],
    auto_class_weights  = 'Balanced',
    eval_metric         = 'AUC',
    random_seed         = SEED,
    verbose             = 100,
    **study.best_trial.params
)
# Eliminar early_stopping_rounds del refit final (ya fijamos n_trees óptimo)
best_params.pop('early_stopping_rounds', None)

model = CatBoostClassifier(**best_params)
model.fit(X, y)

y_prob = model.predict_proba(X)[:, 1]

# CV AUC final (5-fold stratified)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_aucs = []
cv_params = {**best_params, 'verbose': False}
for tr_i, val_i in skf.split(X, y):
    m_fold = CatBoostClassifier(**cv_params)
    m_fold.fit(X[tr_i], y[tr_i])
    cv_aucs.append(roc_auc_score(y[val_i], m_fold.predict_proba(X[val_i])[:, 1]))
cv_auc_final = np.mean(cv_aucs)
cv_auc_std   = np.std(cv_aucs)

metrics = compute_metrics(y, y_prob, threshold=0.5, label='CatBoost')
metrics['CV_AUC']     = round(cv_auc_final, 4)
metrics['CV_AUC_std'] = round(cv_auc_std, 4)
metrics['n_trees']    = best_params['iterations']

print('\nCATBOOST — MÉTRICAS:')
for k, v in metrics.items():
    if k != 'Model':
        print(f'  {k:<14}: {v}')

---
## Visualizaciones

In [ ]:
# Optuna: historial de trials
trial_df = study.trials_dataframe()[['number','value']].rename(columns={'value':'Objetivo'})

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(trial_df['number'], trial_df['Objetivo'], 'o-', color='#e67e22')
ax.axhline(study.best_value, color='#e74c3c', ls='--',
           label=f'Best = {study.best_value:.4f} (trial {study.best_trial.number})')
ax.set_xlabel('Trial'); ax.set_ylabel('Val AUC penalizado')
ax.set_title('Optuna — CatBoost'); ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fig, ax = plt.subplots(figsize=(8, 6))
plot_roc_curve(y, y_prob, label='CatBoost', color='#e67e22', ax=ax,
               title='ROC Curve — CatBoost')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
importance = model.get_feature_importance()
feat_imp_df = pd.DataFrame({'feature': feature_cols, 'importance': importance})
feat_imp_df = feat_imp_df.sort_values('importance', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(feat_imp_df['feature'], feat_imp_df['importance'], color='#e67e22', alpha=0.8)
ax.set_xlabel('Importance'); ax.set_title('Feature Importance — CatBoost (Top 20)')
plt.tight_layout()
plt.show()

In [ ]:
# Learning curve (Train AUC vs Val AUC)
model_lc = CatBoostClassifier(
    **{**best_params, 'iterations': 1000, 'early_stopping_rounds': 50, 'verbose': False}
)
model_lc.fit(X_tr, y_tr, eval_set=(X_val, y_val))
evals = model_lc.get_evals_result()

train_auc_lc = evals['learn']['AUC']
val_auc_lc   = evals['validation']['AUC']
rounds = range(1, len(train_auc_lc) + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rounds, train_auc_lc, color='#3498db', lw=1.5, alpha=0.8, label='Train AUC')
ax.plot(rounds, val_auc_lc,   color='#e74c3c', lw=1.5, label='Val AUC')
ax.axvline(model_lc.get_best_iteration() + 1, color='gray', ls='--', alpha=0.6,
           label=f'Early stop @ {model_lc.get_best_iteration() + 1}')
ax.set_xlabel('Iteración'); ax.set_ylabel('AUC')
ax.set_title('Learning Curve — CatBoost'); ax.legend()
plt.tight_layout()
plt.show()

---
## Predicciones Test

In [ ]:
y_pred_test = model.predict_proba(X_test)[:, 1]
submission = pd.DataFrame({'SK_ID_CURR': sk_ids_test, 'TARGET': y_pred_test})
_sub_path = DATA_DIR / 'submission_12_catboost.csv'
submission.to_csv(_sub_path, index=False)
print(f'Submission guardada: {_sub_path.name}  ({len(submission):,} filas)')
print(f'TARGET stats: mean={y_pred_test.mean():.4f}  min={y_pred_test.min():.4f}  max={y_pred_test.max():.4f}')

---
## Resumen Final

In [ ]:
print('=' * 65)
print('CATBOOST — RESUMEN')
print('=' * 65)
print(f'  learning_rate    : {best_params["learning_rate"]:.5f}')
print(f'  depth            : {best_params["depth"]}')
print(f'  l2_leaf_reg      : {best_params["l2_leaf_reg"]:.4f}')
print(f'  n_trees          : {best_params["iterations"]}')
print(f'  Train AUC        : {metrics["AUC"]}')
print(f'  CV AUC (5-fold)  : {metrics["CV_AUC"]} ± {metrics["CV_AUC_std"]}')
print(f'  Recall           : {metrics["Recall"]}')
print(f'  Precision        : {metrics["Precision"]}')
print(f'  F1               : {metrics["F1"]}')
print('=' * 65)

---
## Kaggle Submission — AUC Test (Public Leaderboard)

Envía el CSV a `home-credit-default-risk` y recupera el AUC del Public LB (~30% del test).

> **Límite**: 5 submissions/día.

In [ ]:
from kaggle import KaggleApi
import time

COMPETITION = 'home-credit-default-risk'
_msg = f'12_catboost | train={metrics["AUC"]:.4f} | cv={metrics["CV_AUC"]:.4f} | trees={best_params["iterations"]}'

def _as_str(x):
    return '' if x is None else str(x)

def _get_score(api, comp, file_name, message, poll=10, timeout=300):
    start = time.time()
    while time.time() - start < timeout:
        subs = api.competition_submissions(comp)
        matched = next(
            (s for s in subs
             if _as_str(getattr(s, 'file_name', None)) == file_name
             and _as_str(getattr(s, 'description', None)) == message),
            subs[0] if subs else None
        )
        if matched:
            pub     = _as_str(getattr(matched, 'public_score',  None))
            status  = _as_str(getattr(matched, 'status',        None))
            elapsed = int(time.time() - start)
            print(f'  [{elapsed:>3}s] status={status!r}  public_score={pub!r}')
            if pub and pub.lower() not in ('', 'none', 'null', '-'):
                priv = _as_str(getattr(matched, 'private_score', None))
                return float(pub), (float(priv) if priv and priv.lower() not in ('','none','null','-') else None)
        time.sleep(poll)
    return None, None

_api = KaggleApi()
_api.authenticate()

print(f'Enviando: {_msg}')
_api.competition_submit(file_name=str(_sub_path), message=_msg, competition=COMPETITION)
print('Esperando scoring (poll 10 s / máx 5 min)...')

public_auc, private_auc = _get_score(_api, COMPETITION, _sub_path.name, _msg)
print(f'\nAUC test  Public  LB : {public_auc}')
print(f'AUC test  Private LB : {private_auc}')